# Time Tensor

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import functools
from functools import partial, singledispatch
from typing import Optional, Union

import numpy as np
import pandas
import torch
from numpy.typing import ArrayLike
from pandas import DataFrame, Index, Series
from pandas.core import indexing
from torch import Tensor

In [3]:
from tsdm.datasets import ETTh1

In [4]:
ds = ETTh1.dataset

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-1.674,3.550,-5.615,2.132,3.472,1.523,10.904000
2018-06-26 16:00:00,-5.492,4.287,-9.132,2.274,3.533,1.675,11.044000
2018-06-26 17:00:00,2.813,3.818,-0.817,2.097,3.716,1.523,10.271000


In [5]:
from pandas.core.indexing import _AtIndexer, _iAtIndexer, _iLocIndexer, _LocIndexer

In [6]:
Tensor(ds.values)

tensor([[ 5.8270,  2.0090,  1.5990,  ...,  4.2030,  1.3400, 30.5310],
        [ 5.6930,  2.0760,  1.4920,  ...,  4.1420,  1.3710, 27.7870],
        [ 5.1570,  1.7410,  1.2790,  ...,  3.7770,  1.2180, 27.7870],
        ...,
        [ 2.8130,  3.8180, -0.8170,  ...,  3.7160,  1.5230, 10.2710],
        [ 9.2430,  3.8180,  5.4720,  ...,  3.6550,  1.4320,  9.7780],
        [10.1140,  3.5500,  6.1830,  ...,  3.7160,  1.4620,  9.5670]])

In [7]:
from torch.utils.data import Dataset, TensorDataset

In [8]:
class IndexMethodClone:
    def __init__(self, data, index_method):
        self.data = data
        self.index_method = index_method

    def __getitem__(self, item):
        print(item)
        idx = self.index_method[item]
        return self.data[idx]

In [20]:
class TimeTensor(Tensor):
    @staticmethod
    def __new__(
        cls,
        x: Union[Tensor, DataFrame, Series, ArrayLike],
        *args,
        index: Optional[Index] = None,
        **kwargs,
    ):
        print(args, kwargs)
        if isinstance(x, DataFrame) or isinstance(x, Series):
            assert index is None, f"Index given, but x is DataFrame/Series"
            x = x.values
        return super().__new__(cls, x, *args, **kwargs)

    def __init__(
        self,
        x: Union[Tensor, DataFrame, Series, ArrayLike],
        index: Optional[Index] = None,
    ):
        super().__init__()  # optional
        if isinstance(x, DataFrame) or isinstance(x, Series):
            index = x.index
        else:
            index = Index(np.arange(len(x))) if index is None else index

        self.index = Series(np.arange(len(x)), index=index)
        self.loc = IndexMethodClone(self, self.index.loc)
        self.iloc = IndexMethodClone(self, self.index.iloc)
        self.at = IndexMethodClone(self, self.index.at)
        self.iat = IndexMethodClone(self, self.index.iat)

In [21]:
ts = TimeTensor(ds)

() {}


tensor([[ 5.8270,  2.0090,  1.5990,  ...,  4.2030,  1.3400, 30.5310],
        [ 5.6930,  2.0760,  1.4920,  ...,  4.1420,  1.3710, 27.7870],
        [ 5.1570,  1.7410,  1.2790,  ...,  3.7770,  1.2180, 27.7870],
        ...,
        [ 2.8130,  3.8180, -0.8170,  ...,  3.7160,  1.5230, 10.2710],
        [ 9.2430,  3.8180,  5.4720,  ...,  3.6550,  1.4320,  9.7780],
        [10.1140,  3.5500,  6.1830,  ...,  3.7160,  1.4620,  9.5670]])

In [19]:
ts.loc["2016":"2017"]

slice('2016', '2017', None)


tensor([[ 5.8270,  2.0090,  1.5990,  ...,  4.2030,  1.3400, 30.5310],
        [ 5.6930,  2.0760,  1.4920,  ...,  4.1420,  1.3710, 27.7870],
        [ 5.1570,  1.7410,  1.2790,  ...,  3.7770,  1.2180, 27.7870],
        ...,
        [14.5350,  3.9520,  9.8080,  ...,  4.5990,  0.8830,  4.2210],
        [14.5350,  3.9520,  9.8080,  ...,  4.5990,  0.8830,  4.2210],
        [14.5350,  3.9520,  9.8080,  ...,  4.5990,  0.8830,  4.2210]])

### TimeTensor Type hint & Type Check

In [12]:
IndexedArray = Union[Series, DataFrame, TimeTensor]


def is_indexed_array(x) -> bool:
    return (
        isinstance(x, Series) or isinstance(x, DataFrame) or isinstance(x, TimeTensor)
    )

## TimeSeriesDataSet

In [78]:
def tensor_info(x: Tensor) -> str:
    return f"{x.__class__.__name__}[{tuple(x.shape)}, {x.dtype}, {x.device.type}]"

In [79]:
tensor_info(torch.randn(1, 2, 3))

'Tensor[(1, 2, 3), torch.float32, cpu]'

In [13]:
from collections.abc import Collection, Iterable, Mapping

In [14]:
class _TupleIndexMethodClone:
    r"""Clone .loc and similar methods to tensor-like object."""

    def __init__(
        self, data: tuple[ArrayLike, ...], index: tuple[Index, ...], method: str = "loc"
    ):
        self.data = data
        self.index = index
        self.method = tuple(getattr(idx, method) for idx in self.index)

    def __getitem__(self, item):
        indices = tuple(method[item] for method in self.method)
        return tuple(data[indices] for data in self.data)

In [95]:
class TimeSeriesDataset(TensorDataset):
    """A general Time Series Dataset.

    Consists of 2 things
    - timeseries: TimeTensor / tuple[TimeTensor]
    - metadata: Tensor / tuple[Tensor]

    When retrieving items, we generally use slices:

    - ds[timestamp] = ds[timestamp:timestamp]
    - ds[t₀:t₁] = tuple[X[t₀:t₁] for X in self.timeseries], metadata
    """

    def __init__(
        self,
        *tensors: IndexedArray,
        observations: Optional[
            Union[
                IndexedArray,
                tuple[Index, ArrayLike],
                dict[Index, ArrayLike],
                Collection[IndexedArray],
                Collection[tuple[Index, ArrayLike]],
            ]
        ] = None,
        metadata: Optional[Union[Tensor, tuple[Tensor]]] = None,
    ):

        ts_tensors = []
        for tensor in tensors:
            ts_tensors.append(TimeTensor(tensor))

        if observations is not None:
            # Case: IndexedArray
            if is_indexed_array(observations):
                ts_tensors.append(TimeTensor(observations))
            # Case: tuple[Index, ArrayLike],
            elif (
                isinstance(observations, tuple)
                and len(observations) == 2
                and isinstance(obseravations[0], Index)
            ):
                index, tensor = obseravations
                ts_tensors.append(TimeTensor(tensor, index=index))
            # Case: dict[Index, ArrayLike],
            elif isinstance(observations, Mapping):
                for index, tensor in observations.items():
                    ts_tensors.append(TimeTensor(tensor, index=index))
            # Case: Iterable
            elif isinstance(observations, Iterable):
                observations = list(observations)
                firstobs = observations[0]
                # Case: Iterable[IndexedArray]
                if is_indexed_array(firstobs):
                    for tensor in observations:
                        ts_tensors.append(TimeTensor(tensor))
                # Case: Iterable[tuple(Index, ArrayLike)]
                elif (
                    isinstance(firstobs, tuple)
                    and len(firstobs) == 2
                    and isinstance(firstobs[0], Index)
                ):
                    for index, tensor in observations:
                        ts_tensors.append(TimeTensor(tensor, index=index))
                else:
                    raise ValueError(f"{observations=} not undertstood")
            else:
                raise ValueError(f"{observations=} not undertstood")

        self.timeseries = tuple(ts_tensors)

        if metadata is not None:
            if isinstance(metadata, tuple):
                self.metadata = tuple(torch.Tensor(tensor) for tensor in metadata)
            else:
                self.metadata = torch.Tensor(metadata)
        else:
            self.metadata = None

    def __repr__(self) -> str:
        pad = r"  "

        if isinstance(self.timeseries, tuple):
            ts_lines = [tensor_info(tensor) for tensor in self.timeseries]
        else:
            ts_lines = [tensor_info(self.timeseries)]

        if self.metadata is None:
            md_lines = [f"{None}"]
        elif isinstance(self.metadata, tuple):
            md_lines = [tensor_info(tensor) for tensor in self.metadata]
        else:
            md_lines = [tensor_info(self.metadata)]

        return (
            f"{self.__class__.__name__}("
            + "".join(["\n" + 2 * pad + line for line in ts_lines])
            + "\n"
            + pad
            + "metadata:"
            + "".join(["\n" + 2 * pad + line for line in md_lines])
            + "\n"
            + ")"
        )

    def __len__(self) -> int:
        r"""The length of the longest timeseries."""
        if isinstance(self.timeseries, tuple):
            return max(len(tensor) for tensor in self.timeseries)
        return len(self.timeseries)

    def __getitem__(self, item):
        r"""Return corresponding slice from each tensor."""
        return tuple(tensor.loc[item] for tensor in self.timeseries)

In [96]:
a = TimeSeriesDataset(ds, ds, ds)

TimeSeriesDataset(
    TimeTensor[(17420, 7), torch.float32, cpu]
    TimeTensor[(17420, 7), torch.float32, cpu]
    TimeTensor[(17420, 7), torch.float32, cpu]
  metadata:
    None
)

In [85]:
a = TimeSeriesDataset(ds, ds, ds)
b = 2

2

In [49]:
ds

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000
...,...,...,...,...,...,...,...
2018-06-26 15:00:00,-1.674,3.550,-5.615,2.132,3.472,1.523,10.904000
2018-06-26 16:00:00,-5.492,4.287,-9.132,2.274,3.533,1.675,11.044000
2018-06-26 17:00:00,2.813,3.818,-0.817,2.097,3.716,1.523,10.271000


In [52]:
torch.randn(5)

tensor([-0.8939, -0.4949, -0.1260, -0.8831,  0.5485])

In [32]:
isinstance(ds, ArrayLike)

TypeError: Subscripted generics cannot be used with class and instance checks

In [37]:
ds["OT"].shape

(17420,)

In [ ]:
class TensorDataset(Dataset[tuple[Tensor, ...]]):
    r"""Dataset wrapping tensors.
    Each sample will be retrieved by indexing tensors along the first dimension.
    Args:
        *tensors (Tensor): tensors that have the same size of the first dimension.
    """
    tensors: tuple[Tensor, ...]

    def __init__(self, *tensors: Tensor) -> None:
        assert all(
            tensors[0].size(0) == tensor.size(0) for tensor in tensors
        ), "Size mismatch between tensors"
        self.tensors = tensors

    def __getitem__(self, index):
        return tuple(tensor[index] for tensor in self.tensors)

    def __len__(self):
        return self.tensors[0].size(0)

In [ ]:
ts.loc["2016-07-01 02:00:00":"2016-07-01 02:00:00"]  # "2016-07-01 03:00:00"]

In [ ]:
ts.index["2016-07-01 02:00:00":"2016-07-01 02:00:00"]

In [ ]:
from pandas import DataFrame

df = DataFrame(np.random.randn(7, 3), index=np.arange(7), columns=["A", "B", "C"])
df.loc[2]

In [ ]:
tuple(range(3))

In [44]:
torch.Tensor((ds.values, ds.values))

tensor([[[ 5.8270,  2.0090,  1.5990,  ...,  4.2030,  1.3400, 30.5310],
         [ 5.6930,  2.0760,  1.4920,  ...,  4.1420,  1.3710, 27.7870],
         [ 5.1570,  1.7410,  1.2790,  ...,  3.7770,  1.2180, 27.7870],
         ...,
         [ 2.8130,  3.8180, -0.8170,  ...,  3.7160,  1.5230, 10.2710],
         [ 9.2430,  3.8180,  5.4720,  ...,  3.6550,  1.4320,  9.7780],
         [10.1140,  3.5500,  6.1830,  ...,  3.7160,  1.4620,  9.5670]],

        [[ 5.8270,  2.0090,  1.5990,  ...,  4.2030,  1.3400, 30.5310],
         [ 5.6930,  2.0760,  1.4920,  ...,  4.1420,  1.3710, 27.7870],
         [ 5.1570,  1.7410,  1.2790,  ...,  3.7770,  1.2180, 27.7870],
         ...,
         [ 2.8130,  3.8180, -0.8170,  ...,  3.7160,  1.5230, 10.2710],
         [ 9.2430,  3.8180,  5.4720,  ...,  3.6550,  1.4320,  9.7780],
         [10.1140,  3.5500,  6.1830,  ...,  3.7160,  1.4620,  9.5670]]])

SyntaxError: invalid syntax (3455386548.py, line 3)